# Image Analyzer Agent

Generic vision agent that takes:

- a **list of image URLs** (any number; comma-separated string is also accepted via the helper below)
- a **context paragraph** describing what the images are about

and returns:

- `analyses`: a list of structured `FigureAnalysis` objects (slug, figure_type, description, x_axis, y_axis, legend, caption, notes, plus the original URL re-attached deterministically)
- `markdown`: a rendered Markdown summary of all figures

## How it works

1. URLs are downloaded into a managed temporary directory (`tempfile.TemporaryDirectory()`) which auto-cleans on exit.
2. Images are split into batches of `batch_size` (default 10).
3. Each batch is sent to the OpenAI Responses API as inline base64 images, with the context paragraph and a per-image caption containing the URL slug.
4. The model returns structured `FigureAnalysis` per image; the agent fills in the `url` field deterministically by mapping `slug` → original URL.
5. Results from all batches are concatenated and rendered as Markdown.

**No experiment-domain knowledge** lives in the agent — it doesn't know about CM1, doesn't group by experiment ID, doesn't auto-build context. The caller flattens its inputs and provides whatever context it wants.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from akd_ext.agents import (
    ImageAnalyzerAgent,
    ImageAnalyzerConfig,
    ImageAnalyzerInputSchema,
    ImageAnalyzerOutputSchema,
)
from akd._base import TextOutput
from IPython.display import Markdown, display
import json

/Users/rsahoo/Desktop/AKD-EXT-Latest/akd-ext/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 1. Provide your inputs

Edit the cell below: paste your URLs (one per line, or comma-separated, or a Python list) and write a short context paragraph describing what the figures show.

In [2]:
# Paste URLs here in any format. Newlines, commas, JSON-style quotes,
# trailing punctuation — all fine. The parser strips them.
raw_urls = """
    "https://s.odsi.io/r76jplr",
    "https://s.odsi.io/1tvcy8z",
    "https://s.odsi.io/9klaxep",
    "https://s.odsi.io/wffbiuc",
    "https://s.odsi.io/reny7bv",
    "https://s.odsi.io/8a5wvqp",
    "https://s.odsi.io/fso6cmw",
    "https://s.odsi.io/rol7550",
    "https://s.odsi.io/8hymmwc",
    "https://s.odsi.io/3vricbg",
    "https://s.odsi.io/wpgavb4",
    "https://s.odsi.io/kc57fdl",
    "https://s.odsi.io/hqf9xu3",
    "https://s.odsi.io/dqh6nyd",
    "https://s.odsi.io/jfa8kqx"
"""

# Stage 1: Gap Agent Output

context = """# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.
"""
import re
# Split on whitespace/commas, then strip stray quotes / brackets / trailing punctuation.
_strip_chars = "\"' ,[](){}"
urls = [
    u.strip().strip(_strip_chars).strip()
    for u in re.split(r"[\s,]+", raw_urls)
    if u.strip()
]
urls = [u for u in urls if u.startswith(("http://", "https://"))]
print(f"{len(urls)} URL(s) prepared")
for u in urls[:5]:
    print(f"  - {u}")
if len(urls) > 5:
    print(f"  … (+{len(urls) - 5} more)")


15 URL(s) prepared
  - https://s.odsi.io/r76jplr
  - https://s.odsi.io/1tvcy8z
  - https://s.odsi.io/9klaxep
  - https://s.odsi.io/wffbiuc
  - https://s.odsi.io/reny7bv
  … (+10 more)


## 2. Run the agent

Defaults: `batch_size=10`, `download_concurrency=8`, `model_name="gpt-5.2"`, `reasoning_effort="medium"`.
With ≤10 URLs you get one batch; ~75 URLs would split into 8 batches.


In [3]:
agent = ImageAnalyzerAgent(
    ImageAnalyzerConfig(
        batch_size=10,            # images per LLM call
        download_concurrency=8,  
        model_name="gpt-5.2",
        reasoning_effort="medium",
    )
)

result = await agent.arun(
    ImageAnalyzerInputSchema(urls=urls, context=context),
)


2026-04-29 14:25:15.710 | WARNING  | akd_ext.agents.image_analyzer:fetch:207 - [ImageAnalyzer] unsupported format for https://s.odsi.io/9klaxep; skipping.
2026-04-29 14:25:16.474 | WARNING  | akd_ext.agents.image_analyzer:fetch:207 - [ImageAnalyzer] unsupported format for https://s.odsi.io/dqh6nyd; skipping.
2026-04-29 14:25:16.934 | INFO     | akd_ext.agents.image_analyzer:_run:299 - [ImageAnalyzer] 13/15 downloads succeeded


## 3. Inspect the structured output

`result.analyses` is `list[FigureAnalysis]`. Each entry is JSON-serialisable via `.model_dump()`.

In [4]:
if isinstance(result, ImageAnalyzerOutputSchema):
    print(f"figures analysed: {len(result.analyses)}")
    print()
    for i, fig in enumerate(result.analyses, 1):
        print(f"--- {i}. {fig.slug} ({fig.figure_type}) ---")
        print(f"  url:         {fig.url}")
        if fig.caption:
            print(f"  caption:     {fig.caption}")
        print(f"  description: {fig.description}")
        if fig.x_axis:
            print(f"  x_axis:      {fig.x_axis}")
        if fig.y_axis:
            print(f"  y_axis:      {fig.y_axis}")
        if fig.legend:
            print(f"  legend:      {fig.legend}")
        if fig.notes:
            print(f"  notes:       {fig.notes}")
        print()
elif isinstance(result, TextOutput):
    print("agent returned a status message:")
    print(result.content)

figures analysed: 13

--- 1. r76jplr (plot) ---
  url:         https://s.odsi.io/r76jplr
  caption:     RQ3 H3.1 — Intensity and structure vs environmental stability
  description: Time series summarizing tropical cyclone intensity and structural evolution for the environmental-stability experiment, including maximum horizontal wind, minimum surface pressure, radius of maximum wind (RMW), and maximum vertical velocity. The curves show rapid intensification followed by fluctuations/plateaus in wind and pressure as convection/structure evolve.
  x_axis:      Time (h), 0–200
  y_axis:      Max horiz. wind (m/s), ~10–105; Min sfc pressure (hPa), ~910–1010; RMW (km), ~0–90; Max vert. velocity (m/s), ~0–28
  legend:      ['output — blue']
  notes:       RMW shows abrupt jumps/spikes early (near the start and ~25 h) before settling into lower values; max vertical velocity has a very large early spike relative to later values.

--- 2. 1tvcy8z (plot) ---
  url:         https://s.odsi.io/1tvcy8z

## 4. Render the Markdown report

In [5]:
if isinstance(result, ImageAnalyzerOutputSchema):
    display(Markdown(result.markdown))

# Image Analysis Report

## Context

# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.

## Figures (13 total)

### 1. `r76jplr` — _plot_

![r76jplr](https://s.odsi.io/r76jplr)

**Caption:** RQ3 H3.1 — Intensity and structure vs environmental stability

**Description:** Time series summarizing tropical cyclone intensity and structural evolution for the environmental-stability experiment, including maximum horizontal wind, minimum surface pressure, radius of maximum wind (RMW), and maximum vertical velocity. The curves show rapid intensification followed by fluctuations/plateaus in wind and pressure as convection/structure evolve.

**X-axis:** Time (h), 0–200

**Y-axis:** Max horiz. wind (m/s), ~10–105; Min sfc pressure (hPa), ~910–1010; RMW (km), ~0–90; Max vert. velocity (m/s), ~0–28

**Notes:** RMW shows abrupt jumps/spikes early (near the start and ~25 h) before settling into lower values; max vertical velocity has a very large early spike relative to later values.

**Legend:**
- output — blue

### 2. `1tvcy8z` — _plot_

![1tvcy8z](https://s.odsi.io/1tvcy8z)

**Caption:** RQ3 H3.1 — 10 m wind and convective depth

**Description:** Time series of near-surface intensity (maximum 10 m wind) alongside convective depth (cloud-top height) under the environmental-stability setup. Wind strengthens over time while cloud-top height rapidly deepens early and then remains roughly steady with small fluctuations.

**X-axis:** Time (h), 0–200

**Y-axis:** Max 10 m wind (m/s), ~10–75; Cloud top height (km), ~0–20

**Notes:** Cloud-top height jumps from near 0 to ~15–20 km within the first ~10 h and then stays near ~15–17 km with minor variability.

**Legend:**
- output — blue

### 3. `wffbiuc` — _plot_

![wffbiuc](https://s.odsi.io/wffbiuc)

**Caption:** RQ3 H3.1 — Theta-e and surface vorticity

**Description:** Thermodynamic and dynamical evolution shown as maximum theta-e below 10 km and maximum surface vorticity for the stability experiment. Theta-e gradually increases with intermittent peaks while surface vorticity ramps up to a maximum and later weakens/levels off.

**X-axis:** Time (h), 0–200

**Y-axis:** Max theta-e below 10 km (K), ~360–379; Max sfc vorticity (1/s), ~0.000–0.020

**Notes:** Surface vorticity rises sharply around ~30–80 h and peaks near ~0.02 1/s before declining to ~0.012–0.015 1/s later.

**Legend:**
- output — blue

### 4. `reny7bv` — _plot_

![reny7bv](https://s.odsi.io/reny7bv)

**Caption:** RQ3 H3.1 — BL wind and PBL depth vs stability

**Description:** Boundary-layer intensity and structure evolution: maximum horizontal wind at the lowest model level and maximum PBL depth versus time for the environmental-stability case. Both increase during intensification, with PBL depth deepening from ~0.5 km toward ~2–3 km.

**X-axis:** Time (h), 0–200

**Y-axis:** Max horiz. wind at lowest level (m/s), ~10–85; Max PBL depth (km), ~0.5–3.0

**Notes:** PBL depth shows step-like increases and noisy variability, especially after ~70 h, with values clustering near ~2.2–2.9 km late.

**Legend:**
- output — blue

### 5. `8a5wvqp` — _plot_

![8a5wvqp](https://s.odsi.io/8a5wvqp)

**Caption:** RQ3 H3.1 — Lowest-model-level wind components vs stability

**Description:** Lowest-model-level wind component diagnostics (u and v extremes) through time for the stability experiment. The plots show strong negative u (minimum u) and large positive v (maximum v) as the circulation intensifies, with notable transient spikes in the component extrema.

**X-axis:** Time (h), 0–200

**Y-axis:** Max u at lowest level (m/s), ~0–14; Min u at lowest level (m/s), ~-50–0; Max v at lowest level (m/s), ~10–80; Min v at lowest level (m/s), ~-12–0

**Notes:** Min v stays near 0 early then exhibits many sharp negative spikes between ~70–140 h (down to roughly -12 m/s), suggesting intermittent asymmetries or sampling spikes.

**Legend:**
- output — blue

### 6. `fso6cmw` — _plot_

![fso6cmw](https://s.odsi.io/fso6cmw)

**Caption:** RQ3 H3.1 — Vertical vorticity at multiple levels vs stability

**Description:** Evolution of maximum vertical vorticity at multiple heights (surface through 5 km) for the environmental-stability experiment. Vorticity increases during storm spin-up across all levels, with later-time reductions and enhanced variability aloft.

**X-axis:** Time (h), 0–200

**Y-axis:** Max vorticity sfc (1/s), ~0.000–0.020; 1 km (1/s), ~0.000–0.020; 2 km (1/s), ~0.000–0.014; 3 km (1/s), ~0.000–0.012; 4 km (1/s), ~0.000–0.013; 5 km (1/s), ~0.000–0.014

**Notes:** Upper-level (4–5 km) vorticity time series are notably spikier than near-surface after ~40 h, with intermittent peaks.

**Legend:**
- output — blue

### 7. `rol7550` — _plot_

![rol7550](https://s.odsi.io/rol7550)

**Caption:** RQ3 H3.1 — Pressure diagnostics vs stability

**Description:** Pressure evolution diagnostics for the stability experiment, including minimum surface pressure, maximum surface pressure, and the implied pressure deficit. Minimum pressure falls steadily during intensification while pressure deficit grows toward ~100 hPa before later fluctuations.

**X-axis:** Time (h), 0–200

**Y-axis:** Min sfc pressure (hPa), ~910–1010; Max sfc pressure (hPa), ~1014.8–1015.6; Pressure deficit (hPa), ~0–105

**Notes:** Max sfc pressure is nearly constant with small noise and a couple sharp spikes near ~70–80 h.

**Legend:**
- output — blue

### 8. `8hymmwc` — _plot_

![8hymmwc](https://s.odsi.io/8hymmwc)

**Caption:** RQ3 H3.1 — Convective metrics vs stability

**Description:** Convective intensity/structure time series for the environmental-stability experiment: maximum and minimum vertical velocity, height of maximum updraft, and cloud-base/top/depth metrics. The early period shows deep convective development (rapid cloud-top rise) and an extreme early updraft spike, followed by more moderate, fluctuating convection.

**X-axis:** Time (h), 0–200

**Y-axis:** Max w (m/s), ~0–28; Min w (m/s), ~-6–0; Height of max w (km), ~0–21; Cloud top (km), ~0–20; Cloud base (km), ~0.0–0.55; Cloud depth (km), ~0–20

**Notes:** Max w has a very large spike early (~10 h) compared with later values; cloud base exhibits step-like plateaus with intermittent jumps.

**Legend:**
- output — blue

### 9. `3vricbg` — _plot_

![3vricbg](https://s.odsi.io/3vricbg)

**Caption:** RQ3 H3.1 — Theta-e and moisture vs stability

**Description:** Thermodynamic and moisture evolution for the stability experiment, showing maximum theta-e (two definitions), maximum water vapor mixing ratio (qv), and minimum lowest-level theta-e. Max theta-e trends upward with time while qv varies within a narrower band and minimum theta-e shows pronounced mid-simulation dips and later recovery.

**X-axis:** Time (h), 0–200

**Y-axis:** Max theta-e < 10 km (K), ~360–379; Max theta-e at lowest level (K), ~360–379; Max qv (g/kg), ~20.5–23.2; Min theta-e at lowest level (K), ~344–356

**Notes:** Min theta-e at lowest level shows a sharp drop around ~70–80 h (down near mid-340s K) before rebounding to ~353–356 K.

**Legend:**
- output — blue

### 10. `wpgavb4` — _plot_

![wpgavb4](https://s.odsi.io/wpgavb4)

**Caption:** RQ3 H3.1 — Precipitation and mass flux vs stability

**Description:** Hydrometeor/overturning diagnostics for the stability experiment: maximum precipitation rate and total upward/downward mass flux. Precipitation intensifies and becomes more sustained later, while mass flux magnitudes increase to order 10^12 (kg m/s) with strong variability.

**X-axis:** Time (h), 0–200

**Y-axis:** Max precip rate (kg/m²/s), ~0.00–0.05; Total upward mass flux (kg m/s), ~0–5×10^12; Total downward mass flux (kg m/s), ~-5×10^12–0

**Notes:** Upward mass flux shows a pronounced peak near ~150 h approaching ~5×10^12; downward mass flux has deep minima (more negative) near ~145–155 h.

**Legend:**
- output — blue

### 11. `kc57fdl` — _plot_

![kc57fdl](https://s.odsi.io/kc57fdl)

**Caption:** RQ3 H3.1 — RMW vs max wind (scatter)

**Description:** Scatter plot relating radius of maximum wind (RMW) to maximum wind speed, used to summarize storm structure/intensity states under the RQ3 H3.1 stability experiment framing. Points cluster at small-to-moderate RMW with high winds, with a few large-RMW cases associated with low winds.

**X-axis:** RMW (km), ~0 to ~90 km

**Y-axis:** Max wind (m/s), ~10 to ~110 m/s

**Notes:** Notable outliers at large RMW (~55–90 km) with very low max winds (~12–20 m/s), while many points cluster near RMW ~10–25 km with max winds ~75–100+ m/s; relationship appears multi-regime rather than a single trend.

**Legend:**
- output — blue

### 12. `hqf9xu3` — _plot_

![hqf9xu3](https://s.odsi.io/hqf9xu3)

**Caption:** RQ3 H3.1 — Intensity change rate vs stability

**Description:** Two time series showing intensity change rates: dVmax/dt and dPmin/dt through the simulation window, relevant for diagnosing how environmental stability might modulate intensification/weakening bursts. Both derivatives fluctuate rapidly around zero with intermittent strong spikes.

**X-axis:** Time (h), 0 to 200 h

**Y-axis:** Left: dVmax/dt (m/s per hr), ~-10 to ~15; Right: dPmin/dt (hPa/hr), ~-10 to ~6

**Notes:** Title references stability, but the plotted x-axis is Time (h) and no stability grouping/labels are shown; derivative series are very noisy with several sharp spikes (e.g., dVmax/dt up to ~14 and down to ~-10; dPmin/dt down near ~-10).

**Legend:**
- output — blue

### 13. `jfa8kqx` — _plot_

![jfa8kqx](https://s.odsi.io/jfa8kqx)

**Caption:** RQ3 H3.1 — Summary bar charts (peak metrics)

**Description:** Four bar charts summarizing peak/endpoint metrics (Peak Vmax, Min Psfc, Peak Wmax, Min RMW) for the RQ3 H3.1 experiment output, intended to compare intensity and structural extrema. Each subplot shows a single category labeled "output," giving one aggregate value per metric rather than a stable/unstable contrast.

**X-axis:** Category ("output"), single bar (no numeric range)

**Y-axis:** Peak Vmax (m/s), ~0 to ~110; Min Psfc (hPa), ~0 to ~900+; Peak Wmax (m/s), ~0 to ~30; Min RMW (km), ~0 to ~2.0

**Notes:** Only one bar/category is present in each panel ("output"), so the figure does not display differences between stable vs unstable soundings; Min Psfc axis tops near ~900+ hPa, suggesting the plotted minimum is close to the axis ceiling.


## 5. Save the output

Two artifacts: structured JSON (machine-consumable) and Markdown (human-readable).

In [6]:
from pathlib import Path

if isinstance(result, ImageAnalyzerOutputSchema):
    json_path = Path("image_analyzer_output.json")
    json_path.write_text(
        json.dumps(
            {
                "context": context.strip(),
                "analyses": [a.model_dump() for a in result.analyses],
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    md_path = Path("image_analyzer_output.md")
    md_path.write_text(result.markdown, encoding="utf-8")
    print(f"Saved JSON to:     {json_path}")
    print(f"Saved Markdown to: {md_path}")

Saved JSON to:     image_analyzer_output.json
Saved Markdown to: image_analyzer_output.md


## 6. (Optional) Smoke test with public images

If you don't have CM1 URLs handy, run this to verify the pipeline works end-to-end.

In [ ]:
smoke_urls = [
    "https://httpbin.org/redirect-to?url=https://httpbin.org/image/png",
    "https://httpbin.org/image/jpeg",
]
smoke_context = "Public stock images. Identify whether each is a plot or illustration and describe briefly."

smoke_result = await agent.arun(
    ImageAnalyzerInputSchema(urls=smoke_urls, context=smoke_context),
)
if isinstance(smoke_result, ImageAnalyzerOutputSchema):
    display(Markdown(smoke_result.markdown))
else:
    print(smoke_result.content)